# Shuffle monolingual data

In [ ]:
import pandas as pd

# load original South Korean monolingal .xlsx, shuffle, and save as .tsv
df = pd.read_excel("../data/monolingual/SK/original/original_monolingual_sk.xlsx")

print(df.columns)

Index(['sk'], dtype='object')


In [8]:
# shuffle (fixed seed for reproducibility)
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# save as TSV
df_shuffled.to_csv("../data/monolingual/SK/original/original_monolingual_sk_shuffled.tsv", sep="\t", index=False)

print("Saved shuffled:", len(df_shuffled))

Saved shuffled: 395622


# Create subsets of synthetic data

In [6]:
import os
import pandas as pd

df = pd.read_csv("../data/synthetic/synthetic_sk_to_nk.tsv", sep="\t")

base_size = 84376
scales = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]

In [7]:
for scale in scales:
    size = min(int(base_size * scale), len(df)) # ensure that the number of sentence pairs in each subset does not exceed the total sentence pairs in the synthetic data
    
    subset = df.iloc[:size]  # nested subsets
    
    output_dir = "../data/synthetic/subsets"
    filename = f"synthetic_sk_to_nk_subset_{scale:.1f}x.tsv"
    path = os.path.join(output_dir, filename)
    
    subset.to_csv(path, sep="\t", index=False)
    
    print(f"{scale}x -> {size} sentences saved")

0.5x -> 42188 sentences saved
1.0x -> 84376 sentences saved
1.5x -> 126564 sentences saved
2.0x -> 168752 sentences saved
2.5x -> 210940 sentences saved
3.0x -> 253128 sentences saved
3.5x -> 295316 sentences saved
4.0x -> 337504 sentences saved
4.5x -> 379692 sentences saved


# Concatenates each synthetic subset with train_filtered.tsv 

In [8]:
import os
import pandas as pd

In [9]:
# paths
train_path = "../data/bilingual/filtered/train_filtered.tsv"
synthetic_dir = "../data/synthetic/subsets"
output_dir = "../data/bilingual/augmented"

os.makedirs(output_dir, exist_ok=True)

In [10]:
# load original training data
train_df = pd.read_csv(train_path, sep="\t")

print("Original train size:", len(train_df))

Original train size: 84376


In [11]:
# scales
scales = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]

In [12]:
for scale in scales:
    # load synthetic subset
    subset_path = os.path.join(synthetic_dir, f"synthetic_sk_to_nk_subset_{scale:.1f}x.tsv")
    
    synth_df = pd.read_csv(subset_path, sep="\t")
    
    # concatenate
    combined_df = pd.concat([train_df, synth_df], ignore_index=True)
    
    # save
    output_path = os.path.join(output_dir, f"train_augmented_{scale:.1f}x.tsv")
    
    combined_df.to_csv(output_path, sep="\t", index=False)
    
    print(f"{scale}x -> {len(combined_df)} sentences saved")

0.5x -> 126564 sentences saved
1.0x -> 168752 sentences saved
1.5x -> 210940 sentences saved
2.0x -> 253128 sentences saved
2.5x -> 295316 sentences saved
3.0x -> 337504 sentences saved
3.5x -> 379692 sentences saved
4.0x -> 421880 sentences saved
4.5x -> 464068 sentences saved
